# Public peer review document histories

Reproduce the study using the accompanying data and analysis code. In Colab, upload the matching reproduction ZIP when prompted.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile
import zipfile

# Use the accompanying repository, or upload its archive in Colab.
explicit = os.environ.get("PEER_REVIEW_REPOSITORY")
candidates = [Path(explicit)] if explicit else [Path.cwd(), Path.cwd().parent, Path.cwd() / "Repository"]
package = next((p.resolve() for p in candidates
                if (p / "scripts/reproduce.py").is_file() and (p / "data/manifest.sha256").is_file()), None)
if package is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Extract the reproduction ZIP and open this notebook from Repository/notebooks, "
                                "or set PEER_REVIEW_REPOSITORY to that Repository directory.") from exc
    print("Upload the accompanying reproduction ZIP.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one reproduction ZIP.")
    archive_path = Path(next(iter(uploaded)))
    destination = Path(tempfile.mkdtemp(prefix="review_histories_")).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        if sum(info.file_size for info in archive.infolist()) > 200_000_000:
            raise ValueError("This is larger than the numerical reproduction package.")
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Archive contains an unsafe path.")
        archive.extractall(destination)
    matches = [p.parent.parent for p in destination.rglob("scripts/reproduce.py")
               if (p.parent.parent / "data/manifest.sha256").is_file()]
    if len(matches) != 1:
        raise ValueError("The archive must contain exactly one reproduction repository.")
    package = matches[0]

expected = {
    "data/manifest.sha256": "c4f5f6b8062355b52d865bee0b04448fcad566f88ba1440c2078a26ca32b7e53",
    "scripts/reproduce.py": "2cc1b430329079c1ff72a4e03111371bf7bb8b0b6df9e3c6dde61ad1bb2ab29a",
}
for relative, digest in expected.items():
    if hashlib.sha256((package / relative).read_bytes()).hexdigest() != digest:
        raise ValueError(f"Notebook and package do not match: {relative}")
print("Matching analysis code and data manifest verified.")


## Analysis

Reconstruct source-defined histories, conditional intervals and exact-text comparisons. The script verifies all distributed input files before calculation.


In [ ]:
import subprocess
import sys

results = package / "results"
subprocess.run([sys.executable, str(package / "scripts/reproduce.py"),
                "--output", str(results)], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(package / "tests"), "-v"], check=True)


In [ ]:
result_archive = package / "document_history_results.zip"
with zipfile.ZipFile(result_archive, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(results.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
try:
    from google.colab import files
except ImportError:
    print("Numerical results:", result_archive)
else:
    files.download(str(result_archive))
